In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from scrapers import MatchScraper, H2HScraper
from scrapers.base_scraper import BaseScraper
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

TEST_EVENT_ID = "8292"  # ESL Pro League Season 20
TEST_MATCH_INDEX = 0    # Első meccs

print(f"🎯 Konfiguráció:")
print(f"   Event ID: {TEST_EVENT_ID}")
print(f"   Match index: {TEST_MATCH_INDEX}")


In [ ]:
# 3. TARGET MATCHES SCRAPING
print("\n" + "="*60)
print("1️⃣ TARGET MATCHES SCRAPING")
print("="*60)

try:
    scraper = MatchScraper(headless=False)
    target_matches = scraper.scrape_event_matches(TEST_EVENT_ID)
    
    if target_matches.empty:
        raise ValueError("❌ Nincs target match!")
    
    # ordinal ragok eltávolítása
    target_matches["date_clean"] = target_matches["date"].apply(lambda x: re.sub(r'(\d+)(st|nd|rd|th)', r'\1', x))

    # dátummá alakítás
    target_matches["date_parsed"] = pd.to_datetime(target_matches["date_clean"], format="%B %d %Y")

    
    print(f"✅ {len(target_matches)} meccs találva")
    display(target_matches.head(3))
    
except Exception as e:
    print(f"❌ Hiba a target matches scraping közben: {e}")

In [ ]:
# 4. MATCH KIVÁLASZTÁSA
print("\n" + "="*60)
print("2️⃣ MATCH KIVÁLASZTÁSA")
print("="*60)

if TEST_MATCH_INDEX >= len(target_matches):
    TEST_MATCH_INDEX = 0
    print(f"⚠️ Index túl nagy, első meccset használom")

selected_match = target_matches.iloc[TEST_MATCH_INDEX]

print(f"🎯 Kiválasztott meccs (index={TEST_MATCH_INDEX}):")
print(f"  Match ID:   {selected_match['match_id']}")
print(f"  Date:       {selected_match['date_parsed']}")
print(f"  Teams:      {selected_match['team_home']} vs {selected_match['team_away']}")
print(f"  Score:      {selected_match['score_home']} - {selected_match['score_away']}")
print(f"  URL:        {selected_match['link']}")

In [ ]:
# 5. MATCH H2H SCRAPING
print("\n" + "="*60)
print("3️⃣ MATCH H2H SCRAPING")
print("="*60)

try:
    match_url = selected_match['link']
    print(f"🔍 Scraping: {match_url}")
    
    scraper = H2HScraper(headless=False)
    h2h_df = scraper.scrape_match_h2h(match_url)
    
    if h2h_df.empty:
        raise ValueError("❌ H2H scraping sikertelen!")
    
    match_h2h = h2h_df.iloc[0]
    
    print(f"✅ H2H data:")
    print(f"  Home team:        {match_h2h['home_team']}")
    print(f"  Away team:        {match_h2h['away_team']}")
    print(f"  H2H wins home:    {match_h2h['wins_home']}")
    print(f"  H2H wins away:    {match_h2h['wins_away']}")
    print(f"  Home win rate:    {match_h2h['home_win_rate']:.3f}")
    print(f"  Home avg rating:  {match_h2h.get('home_team_avg_rating', 'N/A')}")
    print(f"  Away avg rating:  {match_h2h.get('away_team_avg_rating', 'N/A')}")
    
except Exception as e:
    print(f"❌ Hiba a H2H scraping közben: {e}")

In [ ]:
# 6. TEAMS EXTRACTION
print("\n" + "="*60)
print("4️⃣ TEAMS EXTRACTION")
print("="*60)

teams = {
    'home': {
        'team_id': str(match_h2h.get('home_team_id', 'unknown')),
        'team_name': match_h2h['home_team']
    },
    'away': {
        'team_id': str(match_h2h.get('away_team_id', 'unknown')), 
        'team_name': match_h2h['away_team']
    }
}

print(f"✅ Teams extracted:")
print(f"  Home: {teams['home']['team_name']} (ID: {teams['home']['team_id']})")
print(f"  Away: {teams['away']['team_name']} (ID: {teams['away']['team_id']})")

In [ ]:
# 7. TEAM HISTORY SCRAPING HELPER
print("\n" + "="*60)
print("5️⃣ TEAM HISTORY SCRAPING HELPER")
print("="*60)

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from scrapers.base_scraper import BaseScraper

class TeamHistoryScraper(BaseScraper):
    def scrape_team_matches(self, team_id: str, max_matches: int = 20):
        """Team match history scraping"""
        url = f"https://www.hltv.org/results?team={team_id}"
        self._init_driver()
        self.driver.get(url)

        wait = WebDriverWait(self.driver, 20)
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "results-holder")))

        self._random_delay()

        matches = []
        all_sublists = self.driver.find_elements(By.CLASS_NAME, "results-sublist")
        print(f"🔍 Összesen {len(all_sublists)} results-sublist betöltve")

        for sublist in all_sublists:
            if len(matches) >= max_matches:
                break

            try:
                headline = sublist.find_element(By.CLASS_NAME, "standard-headline").text.strip()
                match_date = headline.replace("Results for", "").strip()
            except:
                match_date = None

            match_blocks = sublist.find_elements(By.CLASS_NAME, "result-con")
            for match in match_blocks:
                if len(matches) >= max_matches:
                    break

                try:
                    a_tag = match.find_element(By.TAG_NAME, "a")
                    match_url = a_tag.get_attribute("href")
                    match_id = match_url.split('/')[4]

                    table = a_tag.find_element(By.TAG_NAME, "table")
                    tds = table.find_elements(By.TAG_NAME, "td")

                    team1_name = tds[0].find_element(By.CLASS_NAME, "team").text.strip()
                    team2_name = tds[2].find_element(By.CLASS_NAME, "team").text.strip()

                    score_spans = tds[1].find_elements(By.TAG_NAME, "span")
                    score1 = int(score_spans[0].text.strip())
                    score2 = int(score_spans[1].text.strip())

                    team1_html = tds[0].get_attribute("innerHTML")
                    won = "team-won" in team1_html

                    try:
                        map_text = a_tag.find_element(By.CSS_SELECTOR, ".map-text").text.strip()
                    except:
                        map_text = "bo1"

                    opponent_name = team2_name if team1_name else team1_name

                    matches.append({
                        "team_id": team_id,
                        "match_id": match_id,
                        "match_date": match_date,
                        "opponent_name": opponent_name,
                        "result": "win" if won else "loss",
                        "score_for": score1,
                        "score_against": score2,
                        "map_type": map_text,
                        "link": match_url
                    })

                except Exception as e:
                    print(f"⚠️ Hiba egy meccs feldolgozásánál: {e}")
                    continue

        self.close()
        return pd.DataFrame(matches)

print("✅ TeamHistoryScraper helper kész")

In [ ]:
# 8. TEAM HISTORIES SCRAPING
import re

print("\n" + "="*60)
print("6️⃣ TEAM HISTORIES SCRAPING")
print("="*60)

team_histories = {}

for side in ['home', 'away']:
    team_id = teams[side]['team_id']
    team_name = teams[side]['team_name']
    
    print(f"\n📈 Scraping history: {team_name} (ID: {team_id})")
    
    try:
        scraper = TeamHistoryScraper(headless=False)
        history_df = scraper.scrape_team_matches(team_id, max_matches=100)
        
        if history_df.empty:
            print(f"⚠️ Nincs history adat: {team_name}")
            team_histories[side] = pd.DataFrame()
            continue

        # ordinal ragok eltávolítása
        history_df["date_clean"] = history_df["match_date"].apply(lambda x: re.sub(r'(\d+)(st|nd|rd|th)', r'\1', x))
        # dátummá alakítás
        history_df["date_parsed"] = pd.to_datetime(history_df["date_clean"], format="%B %d %Y")

        # csak a meccs dátuma előttiek (megelőző meccsek)
        history_df = history_df[history_df['date_parsed'] < selected_match['date_parsed']]       

        team_histories[side] = history_df
        
        print(f"✅ {len(history_df)} meccs találva")

        print(f"\n📋 Megelőző 3 meccs:")
        display(history_df.head(3))
        
    except Exception as e:
        print(f"❌ Hiba a history scraping közben: {e}")

In [ ]:
# 9. HISTORICAL H2H SCRAPING
print("\n" + "="*60)
print("7️⃣ HISTORICAL H2H SCRAPING")
print("="*60)

historical_h2h = {}
N_MATCHES = 3

for side in ['home', 'away']:
    team_name = teams[side]['team_name']
    team_id = teams[side]['team_id']
    print(f"\n🔍 {team_name} last {N_MATCHES} matches H2H scraping:")
    
    history = team_histories.get(side, pd.DataFrame())
    
    if history.empty:
        print(f"  ⚠️ Nincs history, skip")
        historical_h2h[side] = {
            'avg_rating': None,
            'avg_adr': None, 
            'avg_swing': None
        }
        continue

    history = history[history['date_parsed'] < selected_match['date_parsed']].reset_index(drop=True)
    
    # Last N match URL-ek
    last_n_matches = history.head(N_MATCHES)
    
    h2h_total = []
    h2h_wins = []
    ratings = []
    rating_stds = []
    adrs = []
    adr_stds = []
    swings = []
    swing_stds = []
    
    for idx, match in last_n_matches.iterrows():
        match_url = match['link']
        print(f"  [{idx+1}/{N_MATCHES}] Scraping: {match['date_parsed'].date()} vs {match['opponent_name']}\n{match_url}")

        try:
            scraper = H2HScraper(headless=False)
            hist_h2h_df = scraper.scrape_match_h2h(match_url)

            if hist_h2h_df.empty:
                raise ValueError("❌ H2H scraping sikertelen!")
            
            if str(hist_h2h_df['home_team_id'][0]) == str(team_id):
                h2h_total.append(hist_h2h_df['total_non_overtime'][0])
                h2h_wins.append(hist_h2h_df['wins_home'][0])
                ratings.append(hist_h2h_df['home_team_avg_rating'][0])
                rating_stds.append(hist_h2h_df['home_team_std_rating'][0])
                adrs.append(hist_h2h_df['home_team_avg_ADR'][0])
                adr_stds.append(hist_h2h_df['home_team_std_ADR'][0])
                swings.append(hist_h2h_df['home_team_avg_Swing'][0])
                swing_stds.append(hist_h2h_df['home_team_std_Swing'][0])
                
            elif str(hist_h2h_df['away_team_id'][0]) == str(team_id):
                h2h_total.append(hist_h2h_df['total_non_overtime'][0])
                h2h_wins.append(hist_h2h_df['wins_away'][0])
                ratings.append(hist_h2h_df['away_team_avg_rating'][0])
                rating_stds.append(hist_h2h_df['away_team_std_rating'][0])
                adrs.append(hist_h2h_df['away_team_avg_ADR'][0])
                adr_stds.append(hist_h2h_df['away_team_std_ADR'][0])
                swings.append(hist_h2h_df['away_team_avg_Swing'][0])
                swing_stds.append(hist_h2h_df['away_team_std_Swing'][0])
            else:
                print("ERROR: team_id not in df")
                        
        except Exception as e:
            print(f"    ⚠️ H2H scraping hiba: {e}")
            continue
   
    # Átlagok számítása
    avg_h2h_winrate = np.sum(h2h_wins) / np.sum(h2h_total) if h2h_total else None
    avg_rating = np.mean(ratings) if ratings else None
    avg_rating_std = np.mean(rating_stds) if rating_stds else None
    avg_adr = np.mean(adrs) if adrs else None
    avg_adr_std = np.mean(adr_stds) if adr_stds else None
    avg_swing = np.mean(swings) if swings else None
    avg_swing_std = np.mean(swing_stds) if swing_stds else None

    historical_h2h[side] = {
        'avg_h2h_winrate': avg_h2h_winrate,
        'avg_rating': avg_rating,
        'avg_rating_std': avg_rating_std,
        'avg_adr': avg_adr,
        'avg_adr_std': avg_adr_std,
        'avg_swing': avg_swing,
        'avg_swing_std': avg_swing_std,
        'n_matches_scraped': len(ratings)
    }

print(f"\n📊 Historical H2H stats összegzés:")
for side in ['home', 'away']:
    stats = historical_h2h[side]
    print(f"  {teams[side]['team_name']}:")
    print(f"    Avg H2H winrate:  {stats['avg_h2h_winrate']:.3f}" if stats['avg_h2h_winrate'] else "    Avg rating:  N/A")
    print(f"    Avg rating:  {stats['avg_rating']:.3f}" if stats['avg_rating'] else "    Avg rating:  N/A")
    print(f"    Avg ADR:     {stats['avg_adr']:.1f}" if stats['avg_adr'] else "    Avg ADR:     N/A")
    print(f"    Avg Swing:   {stats['avg_swing']:.1f}" if stats['avg_swing'] else "    Avg Swing:   N/A")


In [ ]:
# 10. ROLLING FEATURES SZÁMÍTÁSA
print("\n" + "="*60)
print("8️⃣ ROLLING FEATURES SZÁMÍTÁSA")
print("="*60)

ml_input_row = {}

for side in ['home', 'away']:
    team_name = teams[side]['team_name']
    print(f"\n📊 {team_name} rolling features:")
    
    history = team_histories.get(side, pd.DataFrame())
    
    if history.empty:
        print(f"  ⚠️ Nincs history adat")
        continue
    
    # Last 3 winrate
    last_3 = history.head(3)
    last_3_wr = (last_3['result'] == 'win').mean() if len(last_3) > 0 else None
    
    # Last 5 winrate  
    last_5 = history.head(5)
    last_5_wr = (last_5['result'] == 'win').mean() if len(last_5) > 0 else None
    
    # Avg scores
    last_3_avg_for = last_3['score_for'].mean() if len(last_3) > 0 else None
    last_3_avg_against = last_3['score_against'].mean() if len(last_3) > 0 else None
    
    # Current streak
    streak = 0
    if len(history) > 0:
        last_result = history.iloc[0]['result']
        for _, match in history.iterrows():
            if match['result'] == last_result:
                streak += 1
            else:
                break
        if last_result == 'loss':
            streak *= -1
    
    print(f"  Last 3 winrate:    {last_3_wr:.3f}" if last_3_wr else "  Last 3 winrate:    N/A")
    print(f"  Last 5 winrate:    {last_5_wr:.3f}" if last_5_wr else "  Last 5 winrate:    N/A")
    print(f"  Last 3 avg score:  {last_3_avg_for:.1f} - {last_3_avg_against:.1f}" if last_3_avg_for else "  Last 3 avg score:  N/A")
    print(f"  Current streak:    {streak:+d}")
    
    # Store
    ml_input_row[f'{side}_last_3_winrate'] = last_3_wr
    ml_input_row[f'{side}_last_5_winrate'] = last_5_wr
    ml_input_row[f'{side}_last_3_avg_score_for'] = last_3_avg_for
    ml_input_row[f'{side}_last_3_avg_score_against'] = last_3_avg_against
    ml_input_row[f'{side}_current_streak'] = streak

# Difference features
if 'home_last_3_winrate' in ml_input_row and 'away_last_3_winrate' in ml_input_row:
    diff_last3_wr = ml_input_row['home_last_3_winrate'] - ml_input_row['away_last_3_winrate']
    ml_input_row['diff_last_3_winrate'] = diff_last3_wr
    print(f"\n📊 Difference features:")
    print(f"  Diff last 3 WR:    {diff_last3_wr:+.3f}")

In [ ]:
# 11. RANKINGS ÉS EGYÉB FEATURE-ÖK
print("\n" + "="*60)
print("9️⃣ RANKINGS ÉS EGYÉB FEATURE-ÖK")
print("="*60)

# Mock rankings
ml_input_row['home_current_rank'] = 5
ml_input_row['away_current_rank'] = 8  
ml_input_row['home_rank_change'] = -1  # Javult
ml_input_row['away_rank_change'] = 2   # Romlott

print(f"  Home rank: #{ml_input_row['home_current_rank']} (change: {ml_input_row['home_rank_change']:+d})")
print(f"  Away rank: #{ml_input_row['away_current_rank']} (change: {ml_input_row['away_rank_change']:+d})")

# Basic match info
ml_input_row['match_id'] = selected_match['match_id']
ml_input_row['event_id'] = TEST_EVENT_ID
ml_input_row['date'] = selected_match['date']

# Date features
ml_input_row['date_month'] = 9
ml_input_row['date_day'] = 5  # Friday

# Teams
ml_input_row['team_home'] = teams['home']['team_name']
ml_input_row['team_away'] = teams['away']['team_name']

# H2H features
ml_input_row['H2H_winrate_team1'] = match_h2h['home_win_rate']
ml_input_row['H2H_games'] = match_h2h['wins_home'] + match_h2h['wins_away']

# Match info
ml_input_row['match_rounds'] = selected_match['rounds']
ml_input_row['map_veto_decider'] = 1 if selected_match['rounds'] >= 3 else 0

# Player stats
ml_input_row['avg_rating_top3_team1'] = match_h2h.get('home_team_avg_rating', 1.0)

# Historical H2H features
if 'home' in historical_h2h:
    ml_input_row['home_hist_avg_rating'] = historical_h2h['home'].get('avg_rating')
    ml_input_row['home_hist_avg_adr'] = historical_h2h['home'].get('avg_adr')
    ml_input_row['home_hist_avg_swing'] = historical_h2h['home'].get('avg_swing')

if 'away' in historical_h2h:
    ml_input_row['away_hist_avg_rating'] = historical_h2h['away'].get('avg_rating')
    ml_input_row['away_hist_avg_adr'] = historical_h2h['away'].get('avg_adr')
    ml_input_row['away_hist_avg_swing'] = historical_h2h['away'].get('avg_swing')

# Historical difference features
if (ml_input_row.get('home_hist_avg_rating') and 
    ml_input_row.get('away_hist_avg_rating')):
    ml_input_row['hist_diff_rating'] = (
        ml_input_row['home_hist_avg_rating'] - 
        ml_input_row['away_hist_avg_rating']
    )

# Label (score)
ml_input_row['score_home'] = selected_match['score_home']
ml_input_row['score_away'] = selected_match['score_away']
ml_input_row['label_home_win'] = 1 if selected_match['score_home'] > selected_match['score_away'] else 0

print("✅ Feature-ök összegyűjtve!")

In [ ]:
# 12. VÉGEREDMÉNY - ML INPUT ROW
print("\n" + "="*60)
print("🔚 VÉGEREDMÉNY - ML INPUT ROW")
print("="*60)

# DataFrame-ként megjelenítés
ml_df = pd.DataFrame([ml_input_row])

print("📊 DataFrame nézet:")
display(ml_df.T.style.set_caption("ML Input Row - Transposed"))

print("\n📋 Részletes feature-ök:")
for key, value in ml_input_row.items():
    if isinstance(value, float):
        print(f"  {key:35s} = {value:.4f}")
    else:
        print(f"  {key:35s} = {value}")